# LC 198 — House Robber
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Dynamic Programming
**Pattern:** 1D DP — Binary Choice at Each Position

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> At each house make one
binary choice — rob it (gain nums[i], skip the
neighbor) or skip it (keep the best so far).
dp[i] = max(dp[i-2] + nums[i], dp[i-1]).
</div>

## Official Problem Statement

You are a professional robber planning to rob houses
along a street. Each house has a certain amount of
money stashed. The only constraint stopping you from
robbing each of them is that adjacent houses have
security systems connected and it will automatically
contact the police if two adjacent houses were broken
into on the same night.

Given an integer array `nums` representing the amount
of money of each house, return the maximum amount of
money you can rob tonight without alerting the police.

**Example 1:**
```
Input:  nums = [1,2,3,1]
Output: 4
Explanation: Rob house 0 (1) then house 2 (3) = 4
```
**Example 2:**
```
Input:  nums = [2,7,9,3,1]
Output: 12
Explanation: Rob house 0 (2), 2 (9), 4 (1) = 12
```

**Constraints:**
- `1 <= nums.length <= 100`
- `0 <= nums[i] <= 400`

## What This Is Actually Asking

Pick houses to rob so the total money is as large
as possible, but you can never rob two houses that
are next to each other on the street.
You can skip any house — you don't have to rob
every other one.
Find the maximum total across all valid selections.

## Walk Through an Example by Hand

```
nums = [2, 7, 9, 3, 1]
index:  0  1  2  3  4

dp[i] = best loot achievable up to house i

dp[0] = 2          (only house 0)
dp[1] = max(2,7) = 7   (house 0 or house 1, pick best)

dp[2] = max(dp[0]+nums[2], dp[1])
       = max(2+9, 7) = max(11, 7) = 11
         rob house 2 ^^   skip it ^

dp[3] = max(dp[1]+nums[3], dp[2])
       = max(7+3, 11) = max(10, 11) = 11

dp[4] = max(dp[2]+nums[4], dp[3])
       = max(11+1, 11) = max(12, 11) = 12

Answer: 12  (rob houses 0, 2, 4: 2+9+1)
```

## The Picture

```
Houses:  [2]  [7]  [9]  [3]  [1]
          0    1    2    3    4

At house i, you have exactly two options:

  ROB IT:  +nums[i] but must have skipped house i-1
           -> best you had 2 houses ago + nums[i]
           -> dp[i-2] + nums[i]

  SKIP IT: keep whatever was best up to house i-1
           -> dp[i-1]

  dp[i] = max(dp[i-2] + nums[i], dp[i-1])

Only need last two values (prev2, prev1):

  prev2  prev1   nums[i]  -> curr
    2      7       9       max(2+9, 7) = 11
    7      11      3       max(7+3,11) = 11
    11     11      1       max(11+1,11)= 12  <- answer
```

## When To Use This Pattern

- When you see **can't pick adjacent elements**,
  think **dp[i] = max(dp[i-2]+val, dp[i-1])**
- When each position has a binary choice (take or
  skip), think **1D DP with two-variable optimization**
- When the array is circular (House Robber II),
  think **run twice: exclude first, exclude last**
- When only the last two DP values are needed,
  think **drop the array — two variables suffice**

## The Approach

Handle the single-house edge case first.
Set prev2 to the first house's value and prev1 to
the max of the first two houses.
For each remaining house, compute the best result
as the maximum of robbing this house (prev2 + nums[i])
or skipping it (prev1), then slide the two variables
forward and return prev1 at the end.

In [1]:
from typing import List  # type hints for the solution

In [11]:
def test_harness(func):
    tests = [
        # (nums, expected)
        ([1,2,3,1],      4),   # rob 0+2
        ([2,7,9,3,1],   12),   # rob 0+2+4
        ([2,1,1,2],      4),   # rob 0+3
        ([1,2,3,1],      4),
        ([0],            0),   # single zero
        ([5],            5),   # single house
        ([1,1],          1),   # two equal
        ([200,3,140,20,10], 350),  # skip wisely
        ([1,2,3,4,5,6,7,8,9,10], 30),  # alternating
    ]

    passed = 0
    for i, (nums, expected) in enumerate(tests):
        result = func(nums[:])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"nums={nums} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [13]:
def rob(nums: List[int]) -> int:
    """
    Maximize loot from non-adjacent houses.

    At each house choose: rob it (prev2 + nums[i])
    or skip it (prev1). dp[i] = max of those two.
    Only the last two values are needed, so track
    prev2 and prev1 instead of the full array.
    Return prev1 after the last house.

    Time:  O(n) — one pass through the array
    Space: O(1) — two variables only
    """
   
    if len(nums) == 1 : return nums[0]
    if len(nums) == 2 : return max(nums[0], nums[1])
    prev = max(nums[0], nums[1])
    pprev = nums[0]
    curr = 0
    for i in range(2, len(nums)):
        curr = max(nums[i]+ pprev, prev)
        pprev = prev
        prev = curr
    return curr
        
    r'''
4
12
5
4
Test 1: PASSED | nums=[1, 2, 3, 1] | expected=4 | got=4
Test 2: PASSED | nums=[2, 7, 9, 3, 1] | expected=12 | got=12
Test 3: PASSED | nums=[2, 1, 1, 2] | expected=4 | got=4
Test 4: PASSED | nums=[1, 2, 3, 1] | expected=4 | got=4
Test 5: PASSED | nums=[0] | expected=0 | got=0
Test 6: PASSED | nums=[5] | expected=5 | got=5
Test 7: PASSED | nums=[1, 1] | expected=1 | got=1
Test 8: PASSED | nums=[200, 3, 140, 20, 10] | expected=350 | got=350
Test 9: PASSED | nums=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] | expected=30 | got=30

9/9 tests passed
    '''
def test():

    # Quick debug — run this cell while building
    print(rob([1,2,3,1]))      # 4
    print(rob([2,7,9,3,1]))    # 12
    print(rob([5]))             # 5
    print(rob([2,1,1,2]))      # 4
    test_harness(rob)
test()

4
12
5
4
Test 1: PASSED | nums=[1, 2, 3, 1] | expected=4 | got=4
Test 2: PASSED | nums=[2, 7, 9, 3, 1] | expected=12 | got=12
Test 3: PASSED | nums=[2, 1, 1, 2] | expected=4 | got=4
Test 4: PASSED | nums=[1, 2, 3, 1] | expected=4 | got=4
Test 5: PASSED | nums=[0] | expected=0 | got=0
Test 6: PASSED | nums=[5] | expected=5 | got=5
Test 7: PASSED | nums=[1, 1] | expected=1 | got=1
Test 8: PASSED | nums=[200, 3, 140, 20, 10] | expected=350 | got=350
Test 9: PASSED | nums=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10] | expected=30 | got=30

9/9 tests passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(rob)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — try all subsets | O(2^n) | O(n) |
| DP array | O(n) | O(n) |
| Two-variable space-optimized | O(n) | O(1) |

The two-variable version is the interview answer —
dp[i] only ever looks back two steps so the full
array is never needed.

## Real World Connection

At Citi, batch job scheduling often has a similar
constraint: two resource-heavy jobs cannot run in
adjacent time slots because they share the same
compute cluster and would saturate it.
Given a list of estimated processing values per slot,
the House Robber recurrence finds the schedule that
maximises total throughput without triggering a
resource contention alert — the exact same DP.
On AWS, the same pattern applies to Lambda
concurrency budgets: maximize the number of
high-priority functions scheduled while preventing
two back-to-back bursts from hitting the account
concurrency limit.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra